## Data collection

Gets all of the synched properties in the /ToolItems directory

In [36]:
import pathlib as pl
import re
from utils import *
import pprint
import pandas as pd

# Gets the path of every .cs file in the ToolItems directory
project_base_dir = pl.Path(r'C:\Astros\simulator\ASTROS2020-Simulator-Project\Assets')
toolitems_base_dir = project_base_dir / 'ToolItems'
toolitem_file_paths = list(toolitems_base_dir.rglob('*.cs'))

toolitem_properties_map: dict[str, list[str]] = {}
attribute_pattern = re.compile(r'\[GenNetCodeSync')

# Iterates and opens each c# file in tool_items_base_dir
for file_path in toolitem_file_paths:
    with open(file_path, 'r', encoding='Latin-1', errors='ignore') as file:
        # Reads the file line-by-line
        lines = file.readlines()
        for idx, line in enumerate(lines):
            if attribute_pattern.search(line):
                # Look forward to find the property/field declaration
                for next_line in lines[idx + 1:]:
                    stripped = next_line.strip()
                    # Skip empty lines, comments, and further attribute lines
                    if not stripped or stripped.startswith('//') or stripped.startswith('['):
                        continue
                    if file_path.name not in toolitem_properties_map:
                        toolitem_properties_map[file_path.name] = []
                    # toolitem_properties_map[file_path.name].append(stripped)           # appends the entire property declaraytion line
                    stripped = stripped.split()
                    if "new" in stripped:
                        stripped.remove("new")
                    toolitem_properties_map[file_path.name].append(stripped[2])  # appends only the property name        
                    break

print(toolitem_properties_map)
properties_found_count = get_property_count(toolitem_properties_map)
print(f'Found [{properties_found_count}] properties accross [{len(toolitem_properties_map.keys())}] different files')


{'LocalTacticalLine.cs': ['tacticalType'], 'LocalMovementRestrictiveArea.cs': ['type', 'parts', 'name', 'color', 'layerId', 'editingClientId'], 'LocalInterestPoint.cs': ['type', 'geoData'], 'LocalItinerary.cs': ['points'], 'LocalToolItem.cs': ['toolItemType', 'name', 'color', 'layerId', 'editingClientId', 'flags'], 'LocalRectangle.cs': ['renderType', 'geoData'], 'LocalMultipartPolygon.cs': ['parts', 'renderType'], 'LocalLine.cs': ['controlPoints', 'renderType', 'arrowheadType'], 'LocalAbstractEllipse.cs': ['renderType', 'geoData'], 'LocalArrow.cs': ['renderType', 'arrowData'], 'LocalMeasure.cs': ['name', 'period', 'emitter', 'observation'], 'LocalShapedMeasure.cs': ['type', 'color', 'layerId', 'editingClientId'], 'LocalAreaShapedMeasure.cs': ['points'], 'LocalLineShapedMeasure.cs': ['points'], 'LocalInterdictionCell.cs': ['cellType', 'geoCenter', 'geoSize', 'degrees', 'ceilingAltitudeInMeters', 'separationAltitudeInMeters'], 'LocalAirspaceZone.cs': ['zoneType', 'widthInMeters', 'points

Let's make that 60 properties to get an even number an also be in line with what the paper says. LocalAirspaceZone (3), and LocalInterdictionCell (6)

In [37]:
toolitem_properties_map.pop('LocalAirspaceZone.cs')
toolitem_properties_map.pop('LocalInterdictionCell.cs')

properties_found_count = get_property_count(toolitem_properties_map)
print(f'Found [{properties_found_count}] properties accross [{len(toolitem_properties_map.keys())}] different files')

Found [60] properties accross [21] different files


Now that we have all of the classes and their properties beeing synched, we can now get the metrcis in a more organized fashion. The metrics being gathered are:
1. Development effort
    - LOC written by **property**
    - Number of files that need to be modified by **class**
2. Coupling
    - Number of times that the local **class** references the net one
    - Number of times that the SRP is violated for each **property**
3. Quality
    - Percentage of duplicated code in a **class**
    - Cyclomatic complexity for each **class**
4. Proccess:
    - Compilation time

All fo the metrics will be collected in this [toolitem: metrics] dict

In [38]:
toolitem_metrics_map: dict[str, dict[str, any]] = {}

for toolitem, properties in toolitem_properties_map.items():
    toolitem_metrics_map[toolitem] = {}
    toolitem_metrics_map[toolitem]['properties'] = {}
    for prop in properties:
        prop_metrics = {'name': prop}
        toolitem_metrics_map[toolitem]['properties'][prop] = {}
print(toolitem_metrics_map)

{'LocalTacticalLine.cs': {'properties': {'tacticalType': {}}}, 'LocalMovementRestrictiveArea.cs': {'properties': {'type': {}, 'parts': {}, 'name': {}, 'color': {}, 'layerId': {}, 'editingClientId': {}}}, 'LocalInterestPoint.cs': {'properties': {'type': {}, 'geoData': {}}}, 'LocalItinerary.cs': {'properties': {'points': {}}}, 'LocalToolItem.cs': {'properties': {'toolItemType': {}, 'name': {}, 'color': {}, 'layerId': {}, 'editingClientId': {}, 'flags': {}}}, 'LocalRectangle.cs': {'properties': {'renderType': {}, 'geoData': {}}}, 'LocalMultipartPolygon.cs': {'properties': {'parts': {}, 'renderType': {}}}, 'LocalLine.cs': {'properties': {'controlPoints': {}, 'renderType': {}, 'arrowheadType': {}}}, 'LocalAbstractEllipse.cs': {'properties': {'renderType': {}, 'geoData': {}}}, 'LocalArrow.cs': {'properties': {'renderType': {}, 'arrowData': {}}}, 'LocalMeasure.cs': {'properties': {'name': {}, 'period': {}, 'emitter': {}, 'observation': {}}}, 'LocalShapedMeasure.cs': {'properties': {'type': {}

#### 1) Development effort
1. LOC written by **property**
2. Number of files that need to be modified by **class**

In [39]:
# LOC written by property
for toolitem, metrics in toolitem_metrics_map.items():
    file_path = next((p for p in toolitem_file_paths if p.name == toolitem), None)
    if not file_path:
        continue
        
    with open(file_path, 'r', encoding='Latin-1', errors='ignore') as f:
        lines = f.readlines()
        
    for prop, prop_metrics in metrics['properties'].items():
        i = 0
        while i < len(lines):
            line = lines[i].strip()
            if line.startswith('[GenNetCodeSync'):
                loc = 0
                j = i
                prop_found = False
                brace_count = 0
                seen_brace = False
                
                while j < len(lines):
                    curr_line = lines[j].strip()
                    if curr_line and not curr_line.startswith('//'):
                        loc += 1
                        
                        if prop in curr_line:
                            if prop.isalnum():
                                if re.search(rf'\b{prop}\b', curr_line):
                                    prop_found = True
                            else:
                                prop_found = True
                        
                        brace_count += curr_line.count('{')
                        brace_count -= curr_line.count('}')
                        if '{' in curr_line:
                            seen_brace = True
                            
                        ended = False
                        if seen_brace and brace_count == 0:
                            ended = True
                        elif not seen_brace and ';' in curr_line:
                            ended = True
                            
                        if ended:
                            if prop_found:
                                prop_metrics['LOC'] = loc
                            break
                    j += 1
                
                if prop_found:
                    break
            i += 1

# Number of files that need to be modified by class
for toolitem in toolitem_metrics_map.keys():
    toolitem_metrics_map[toolitem]['file_count'] = 1  # always 1 post-refactor

pprint.pprint(toolitem_metrics_map['LocalToolItem.cs'])

{'file_count': 1,
 'properties': {'color': {'LOC': 4},
                'editingClientId': {'LOC': 4},
                'flags': {'LOC': 4},
                'layerId': {'LOC': 4},
                'name': {'LOC': 4},
                'toolItemType': {'LOC': 4}}}


#### 2) Coupling
- Number of times that the local **class** references the net one
- Number of times that the SRP is violated for each **property**

In [40]:
# 2) Coupling - Number of times the local class references the net one
for toolitem in toolitem_metrics_map.keys():
    file_path = next((p for p in toolitem_file_paths if p.name == toolitem), None)
    if not file_path:
        continue
    # Class name derived from filename, replacing 'Local' with 'Net'
    local_class = file_path.stem
    net_class = local_class.replace('Local', 'Net', 1)
    with open(file_path, 'r', encoding='Latin-1', errors='ignore') as f:
        content = f.read()
    # Match exact word occurrences (avoids partial substring matches)
    net_references = len(re.findall(rf'\b{net_class}\b', content))
    toolitem_metrics_map[toolitem]['net_references'] = net_references
    
# Print summary
pprint.pprint({toolitem: metrics['net_references'] for toolitem, metrics in toolitem_metrics_map.items()})
# print(f'LocalToolItem: {toolitem_metrics_map['LocalToolItem.cs']['net_references']}')

{'LocalAbstractEllipse.cs': 0,
 'LocalAirspaceDefenseVolume.cs': 0,
 'LocalAirspaceRoute.cs': 0,
 'LocalAirspaceTrajectory.cs': 0,
 'LocalAirspaceVolume.cs': 0,
 'LocalAreaShapedMeasure.cs': 0,
 'LocalArrow.cs': 0,
 'LocalBoundary.cs': 0,
 'LocalElementsBoundary.cs': 0,
 'LocalInterestPoint.cs': 0,
 'LocalItinerary.cs': 0,
 'LocalLine.cs': 0,
 'LocalLineShapedMeasure.cs': 0,
 'LocalMeasure.cs': 0,
 'LocalMovementRestrictiveArea.cs': 0,
 'LocalMultipartPolygon.cs': 0,
 'LocalRectangle.cs': 0,
 'LocalRestrictiveOperationZone.cs': 0,
 'LocalShapedMeasure.cs': 0,
 'LocalTacticalLine.cs': 0,
 'LocalToolItem.cs': 0}


By consequence, the number of SRP violations for each of the properties of all of the toolitems is also 0

In [41]:
for toolitem, metrics in toolitem_metrics_map.items():
    metrics['properties'].pop('SRP_violations', None)  # removes accidental key from earlier
    for prop, prop_metrics in metrics['properties'].items():
        prop_metrics['SRP_violations'] = 0

pprint.pprint(toolitem_metrics_map['LocalToolItem.cs'])

{'file_count': 1,
 'net_references': 0,
 'properties': {'color': {'LOC': 4, 'SRP_violations': 0},
                'editingClientId': {'LOC': 4, 'SRP_violations': 0},
                'flags': {'LOC': 4, 'SRP_violations': 0},
                'layerId': {'LOC': 4, 'SRP_violations': 0},
                'name': {'LOC': 4, 'SRP_violations': 0},
                'toolItemType': {'LOC': 4, 'SRP_violations': 0}}}


#### 3) Quality
- Percentage of duplicated code in a **class**
- **class**:
    - Maintainability Index: Measures ease of code maintenance. Higher values are better.
    - Cyclomatic Complexity: Measures number of branches. Lowe values are better.
    - Class Coupling: Measures number of classes that are referenced. Lower values are better.

In [42]:
# Cyclomatic complexity
# toolitem_properties_map.pop('LocalAirspaceZone.cs')
# toolitem_properties_map.pop('LocalInterdictionCell.cs')

toolitem_metrics_map['LocalAbstractEllipse.cs']['cyclomatic_complexity'] = 20
toolitem_metrics_map['LocalAbstractEllipse.cs']['maintainability_index'] = 75
toolitem_metrics_map['LocalAbstractEllipse.cs']['class_coupling'] = 27

toolitem_metrics_map['LocalAirspaceDefenseVolume.cs']['cyclomatic_complexity'] = 15
toolitem_metrics_map['LocalAirspaceDefenseVolume.cs']['maintainability_index'] = 74
toolitem_metrics_map['LocalAirspaceDefenseVolume.cs']['class_coupling'] = 23

toolitem_metrics_map['LocalAirspaceRoute.cs']['cyclomatic_complexity'] = 47
toolitem_metrics_map['LocalAirspaceRoute.cs']['maintainability_index'] = 73
toolitem_metrics_map['LocalAirspaceRoute.cs']['class_coupling'] = 41

toolitem_metrics_map['LocalAirspaceTrajectory.cs']['cyclomatic_complexity'] = 28
toolitem_metrics_map['LocalAirspaceTrajectory.cs']['maintainability_index'] = 79
toolitem_metrics_map['LocalAirspaceTrajectory.cs']['class_coupling'] = 35

toolitem_metrics_map['LocalAirspaceVolume.cs']['cyclomatic_complexity'] = 49
toolitem_metrics_map['LocalAirspaceVolume.cs']['maintainability_index'] = 74
toolitem_metrics_map['LocalAirspaceVolume.cs']['class_coupling'] = 41

toolitem_metrics_map['LocalAreaShapedMeasure.cs']['cyclomatic_complexity'] = 22
toolitem_metrics_map['LocalAreaShapedMeasure.cs']['maintainability_index'] = 79
toolitem_metrics_map['LocalAreaShapedMeasure.cs']['class_coupling'] = 33

toolitem_metrics_map['LocalArrow.cs']['cyclomatic_complexity'] = 26
toolitem_metrics_map['LocalArrow.cs']['maintainability_index'] = 75
toolitem_metrics_map['LocalArrow.cs']['class_coupling'] = 41

toolitem_metrics_map['LocalBoundary.cs']['cyclomatic_complexity'] = 57
toolitem_metrics_map['LocalBoundary.cs']['maintainability_index'] = 77
toolitem_metrics_map['LocalBoundary.cs']['class_coupling'] = 40

toolitem_metrics_map['LocalElementsBoundary.cs']['cyclomatic_complexity'] = 18
toolitem_metrics_map['LocalElementsBoundary.cs']['maintainability_index'] = 75
toolitem_metrics_map['LocalElementsBoundary.cs']['class_coupling'] = 20

toolitem_metrics_map['LocalInterestPoint.cs']['cyclomatic_complexity'] = 19
toolitem_metrics_map['LocalInterestPoint.cs']['maintainability_index'] = 76
toolitem_metrics_map['LocalInterestPoint.cs']['class_coupling'] = 25

toolitem_metrics_map['LocalItinerary.cs']['cyclomatic_complexity'] = 23
toolitem_metrics_map['LocalItinerary.cs']['maintainability_index'] = 80
toolitem_metrics_map['LocalItinerary.cs']['class_coupling'] = 29

toolitem_metrics_map['LocalLine.cs']['cyclomatic_complexity'] = 32
toolitem_metrics_map['LocalLine.cs']['maintainability_index'] = 76
toolitem_metrics_map['LocalLine.cs']['class_coupling'] = 36

toolitem_metrics_map['LocalLineShapedMeasure.cs']['cyclomatic_complexity'] = 14
toolitem_metrics_map['LocalLineShapedMeasure.cs']['maintainability_index'] = 77
toolitem_metrics_map['LocalLineShapedMeasure.cs']['class_coupling'] = 26

toolitem_metrics_map['LocalMeasure.cs']['cyclomatic_complexity'] = 51
toolitem_metrics_map['LocalMeasure.cs']['maintainability_index'] = 76
toolitem_metrics_map['LocalMeasure.cs']['class_coupling'] = 32

toolitem_metrics_map['LocalMovementRestrictiveArea.cs']['cyclomatic_complexity'] = 60
toolitem_metrics_map['LocalMovementRestrictiveArea.cs']['maintainability_index'] = 78
toolitem_metrics_map['LocalMovementRestrictiveArea.cs']['class_coupling'] = 49

toolitem_metrics_map['LocalMultipartPolygon.cs']['cyclomatic_complexity'] = 20
toolitem_metrics_map['LocalMultipartPolygon.cs']['maintainability_index'] = 77
toolitem_metrics_map['LocalMultipartPolygon.cs']['class_coupling'] = 33

toolitem_metrics_map['LocalRectangle.cs']['cyclomatic_complexity'] = 20
toolitem_metrics_map['LocalRectangle.cs']['maintainability_index'] = 75
toolitem_metrics_map['LocalRectangle.cs']['class_coupling'] = 27

toolitem_metrics_map['LocalRestrictiveOperationZone.cs']['cyclomatic_complexity'] = 12
toolitem_metrics_map['LocalRestrictiveOperationZone.cs']['maintainability_index'] = 74
toolitem_metrics_map['LocalRestrictiveOperationZone.cs']['class_coupling'] = 25

toolitem_metrics_map['LocalShapedMeasure.cs']['cyclomatic_complexity'] = 52
toolitem_metrics_map['LocalShapedMeasure.cs']['maintainability_index'] = 78
toolitem_metrics_map['LocalShapedMeasure.cs']['class_coupling'] = 36

toolitem_metrics_map['LocalTacticalLine.cs']['cyclomatic_complexity'] = 12
toolitem_metrics_map['LocalTacticalLine.cs']['maintainability_index'] = 77
toolitem_metrics_map['LocalTacticalLine.cs']['class_coupling'] = 24

toolitem_metrics_map['LocalToolItem.cs']['cyclomatic_complexity'] = 54
toolitem_metrics_map['LocalToolItem.cs']['maintainability_index'] = 76
toolitem_metrics_map['LocalToolItem.cs']['class_coupling'] = 37

print(len(toolitem_metrics_map.keys()))

21


### 4) Proccess:
- Compilation time


In [43]:
toolitem_metrics_map['compilation_times'] = [12.623, 11.795, 12.818, 12.646, 12.412, 12.468, 13.148, 13.334, 12.490, 12.489]